# 06 - Fine-tune Tashkeel-700M (LFM2 Decoder-Only)

**Base Model:** `Etherll/Tashkeel-700M`
**Architecture:** LFM2 Decoder-Only (742M params)
**Training:** TRL/SFT on Tashkeela dataset

## Why Decoder-Only for Diacritization?
- **Chat-style interface**: Natural instruction following
- **Modern architecture**: LFM2 is optimized for efficiency
- **SFT-trained**: Already fine-tuned with TRL

## Training Techniques:
1. **LoRA**: Efficient fine-tuning (trainable params << total params)
2. **Chat Template**: Proper instruction formatting
3. **SFT Trainer**: Optimized for instruction-following

## Tasks:
1. Load and preprocess KSAA training data
2. Fine-tune with LoRA + SFT
3. Evaluate on dev set
4. Generate test submission

In [1]:
!pip install -q transformers torch accelerate datasets tqdm pandas peft trl bitsandbytes

In [2]:
# Setup
import os, sys, json, re, torch, gc, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Environment detection
IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

# Import utilities (includes safeguards for empty responses & separated characters)
from utils_diacritization import (
    normalize_arabic, remove_diacritics, postprocess_diacritics,
    compute_metrics, print_metrics, sort_by_length,
    is_valid_output, repair_output, get_safe_generation_config
)

# Paths
DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
TRAIN_DIR = DATA_DIR / 'Train'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'tashkeel_700m_ft'

for d in [OUTPUT_DIR, SUBMISSION_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated")

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.6 GB)


## 1. Load Training Data

In [3]:
# Load training data
train_tsv = TRAIN_DIR / 'train_list.tsv'
train_df = pd.read_csv(train_tsv, sep='\t')
print(f"Training samples in TSV: {len(train_df)}")

def load_training_data():
    data = []
    text_dir = TRAIN_DIR / 'train_text'
    
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Loading data"):
        txt_file = text_dir / f"{row['stem']}.txt"
        if txt_file.exists():
            with open(txt_file, 'r', encoding='utf-8') as f:
                diacritized = f.read().strip()
            
            diacritized = normalize_arabic(diacritized)
            undiacritized = remove_diacritics(diacritized)
            
            if len(undiacritized.split()) < 2:
                continue
                
            data.append({
                'id': row['stem'],
                'text_undiacritized': undiacritized,
                'text_diacritized': diacritized
            })
    return data

train_data = load_training_data()
print(f"Loaded {len(train_data)} training samples")

Training samples in TSV: 2327


Loading data: 100%|██████████| 2327/2327 [00:05<00:00, 395.69it/s]

Loaded 2317 training samples


In [4]:
# Create chat-style training data
SYSTEM_PROMPT = "أنت مساعد متخصص في تشكيل النصوص العربية. أضف الحركات (الفتحة، الضمة، الكسرة، السكون، الشدة، التنوين) إلى النص العربي المُدخل."

def format_as_chat(sample):
    """Format sample as chat conversation for SFT."""
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f"أضف التشكيل إلى النص التالي:\n{sample['text_undiacritized']}"},
            {'role': 'assistant', 'content': sample['text_diacritized']}
        ]
    }

chat_data = [format_as_chat(s) for s in train_data]
print(f"Formatted {len(chat_data)} samples as chat conversations")

# Show example
print("\nExample conversation:")
for msg in chat_data[0]['messages']:
    print(f"  [{msg['role']}]: {msg['content'][:50]}...")

Formatted 2317 samples as chat conversations

Example conversation:
  [system]: أنت مساعد متخصص في تشكيل النصوص العربية. أضف الحرك...
  [user]: أضف التشكيل إلى النص التالي:
﻿اخي شعبان القبائل عب...
  [assistant]: ﻿اَخِي شَعْبَانْ الْقَبَائِلْ عَبْرَ التَّارِيخْ ك...


## 2. Load Model with LoRA

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'Etherll/Tashkeel-700M'
MODEL_KEY = 'tashkeel_700m_ft'

print(f"Loading {MODEL_NAME}...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Quantization config - use FP16 compute dtype (not BF16) to avoid AMP issues
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # FP16 for compatibility
    bnb_4bit_use_double_quant=True,
)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if device == "cuda" else None,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16,  # Consistent FP16
)

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Loading Etherll/Tashkeel-700M...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded: 421,625,088 parameters


In [6]:
# Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

Trainable params: 688,128 (0.16%)


## 3. Prepare Dataset

In [7]:
from datasets import Dataset

def apply_chat_template(examples):
    """Apply chat template to messages."""
    texts = []
    for messages in examples['messages']:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {'text': texts}

# Create dataset
train_dataset = Dataset.from_list(chat_data)
train_dataset = train_dataset.map(
    apply_chat_template,
    batched=True,
    remove_columns=['messages'],
    desc="Applying chat template"
)

print(f"Training dataset: {len(train_dataset)} samples")
print(f"\nExample formatted text:\n{train_dataset[0]['text'][:500]}...")

Applying chat template:   0%|          | 0/2317 [00:00<?, ? examples/s]

Training dataset: 2317 samples

Example formatted text:
<|startoftext|><|im_start|>system
أنت مساعد متخصص في تشكيل النصوص العربية. أضف الحركات (الفتحة، الضمة، الكسرة، السكون، الشدة، التنوين) إلى النص العربي المُدخل.<|im_end|>
<|im_start|>user
أضف التشكيل إلى النص التالي:
﻿اخي شعبان القبائل عبر التاريخ كانت دائما تقترب من الحاكم اثنين العشائر ما هي متدينة بمعني يعني ايمانها عفوي<|im_end|>
<|im_start|>assistant
﻿اَخِي شَعْبَانْ الْقَبَائِلْ عَبْرَ التَّارِيخْ كَانَتْ دَائِمًا تَقْتَرِبْ مِنَ الْحَاكِمْ اثْنِينْ الْعَشَائِرْ مَا هِيَ مُتَدَيِّنَةْ بِمَع...


## 4. Training with SFT Trainer

In [9]:
from trl import SFTTrainer, SFTConfig

# SFT Configuration - Optimized for 24GB GPU (4-bit quantized)
# NOTE: With 4-bit quantization, we disable fp16/bf16 to avoid AMP conflicts
sft_config = SFTConfig(
    output_dir=str(CHECKPOINTS_DIR),
    
    # Training - Can use larger batches with 4-bit quantization
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # Effective batch: 16
    
    # Optimizer
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    
    # Precision - Disable AMP since 4-bit already handles precision
    fp16=False,
    bf16=False,
    
    # Saving
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    
    # Logging
    logging_steps=25,
    report_to="none",
    
    # SFT specific
    packing=False,
    dataset_text_field="text",
    
    # Performance
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
)

print("SFT Trainer configured")
print(f"  Epochs: {sft_config.num_train_epochs}")
print(f"  Batch size: {sft_config.per_device_train_batch_size}")
print(f"  Gradient accum: {sft_config.gradient_accumulation_steps}")
print(f"  Effective batch: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Precision: 4-bit quantized (no AMP)")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/2317 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2317 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2317 [00:00<?, ? examples/s]

SFT Trainer configured
  Epochs: 3
  Batch size: 4
  Gradient accum: 4
  Effective batch: 16
  Precision: 4-bit quantized (no AMP)


In [10]:
# Train!
clear_gpu()

# Check for checkpoint
checkpoint = None
checkpoints = list(CHECKPOINTS_DIR.glob('checkpoint-*'))
if checkpoints:
    checkpoint = str(max(checkpoints, key=lambda x: int(x.name.split('-')[1])))
    print(f"Resuming from: {checkpoint}")

print("Starting training...")
trainer.train(resume_from_checkpoint=checkpoint)

GPU Memory: 0.70 GB allocated
Starting training...


Step,Training Loss
25,3.405800
50,2.192895
75,1.343314
100,1.028855
125,0.895120
150,0.794675
175,0.746432
200,0.713488
225,0.710855
250,0.697899


TrainOutput(global_step=435, training_loss=1.012125236686619, metrics={'train_runtime': 409.8808, 'train_samples_per_second': 16.959, 'train_steps_per_second': 1.061, 'total_flos': 6663344895553536.0, 'train_loss': 1.012125236686619})

In [11]:
# Save final model (LoRA weights)
final_path = CHECKPOINTS_DIR / 'final'
model.save_pretrained(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"LoRA weights saved to: {final_path}")
clear_gpu()

LoRA weights saved to: ./checkpoints/tashkeel_700m_ft/final
GPU Memory: 0.72 GB allocated


## 5. Inference Functions

In [12]:
# Load fine-tuned model for inference (FP16 for speed)
from peft import PeftModel

final_model_path = CHECKPOINTS_DIR / 'final'

if final_model_path.exists():
    print(f"Loading fine-tuned LoRA model from {final_model_path}")
    
    # Clear training model first
    del model
    clear_gpu()
    
    # Load base model in FP16 (no quantization for faster inference)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    
    # Load LoRA weights
    model = PeftModel.from_pretrained(base_model, str(final_model_path))
    model = model.merge_and_unload()  # Merge for faster inference
    tokenizer = AutoTokenizer.from_pretrained(str(final_model_path))
else:
    print("Using model from training")

model.eval()
print("Model ready for inference (FP16, merged LoRA)")

Loading fine-tuned LoRA model from ./checkpoints/tashkeel_700m_ft/final
GPU Memory: 0.72 GB allocated


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model ready for inference (FP16, merged LoRA)


In [13]:
@torch.inference_mode()
def diacritize(text: str, apply_postprocess: bool = True) -> str:
    """
    Diacritize Arabic text using the fine-tuned decoder-only model.
    Includes safeguards against empty responses and separated characters.
    """
    original_text = text
    text = normalize_arabic(text)
    
    # Format as chat
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f"أضف التشكيل إلى النص التالي:\n{text}"}
    ]
    
    # Apply chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Get safe generation config
    gen_config = get_safe_generation_config()
    gen_config['pad_token_id'] = tokenizer.pad_token_id
    gen_config['eos_token_id'] = tokenizer.eos_token_id
    
    # Generate with safe parameters
    outputs = model.generate(
        **inputs,
        **gen_config
    )
    
    # Decode only new tokens
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # Repair output (fixes separated characters, validates, falls back to input if invalid)
    if apply_postprocess:
        result = repair_output(result, original_text, fallback_to_input=True)
    
    return result

## 6. Evaluate on Dev Set

In [14]:
# Load dev data
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)

# Checkpoint support
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

# Run inference
processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids:
        continue
    try:
        result = diacritize(item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_checkpoint(dev_predictions)
    except Exception as e:
        print(f"Error on {item['id']}: {e}")
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

# Save predictions
with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

Dev: 260 samples, 0 already done


Dev Set: 100%|██████████| 260/260 [07:22<00:00,  1.70s/it]


In [15]:
# Compute metrics
print("\n" + "="*60)
print("DEV SET RESULTS")
print("="*60)

metrics_with_irab = compute_metrics(dev_predictions, dev_output, exclude_irab=False)
print(f"\n[Including I'rab]")
print(f"  DER: {metrics_with_irab['DER']*100:.2f}%")
print(f"  WER: {metrics_with_irab['WER']*100:.2f}% (PRIMARY)")
print(f"  SER: {metrics_with_irab['SER']*100:.2f}%")

metrics_no_irab = compute_metrics(dev_predictions, dev_output, exclude_irab=True)
print(f"\n[Excluding I'rab]")
print(f"  DER: {metrics_no_irab['DER']*100:.2f}%")
print(f"  WER: {metrics_no_irab['WER']*100:.2f}%")


DEV SET RESULTS

[Including I'rab]
  DER: 27.59%
  WER: 62.14% (PRIMARY)
  SER: 98.85%

[Excluding I'rab]
  DER: 27.10%
  WER: 62.14%


## 7. Generate Test Submission

In [16]:
# Test set inference
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed_ids)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids:
        continue
    try:
        result = diacritize(item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_test_checkpoint(test_predictions)
    except Exception as e:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

# Save predictions
with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

Test: 328 samples, 0 already done


Test Set: 100%|██████████| 328/328 [15:39<00:00,  2.86s/it]


In [17]:
# Create submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print("="*60)
print("SUBMISSION READY")
print("="*60)
print(f"ZIP: {zip_file}")
print(f"Size: {zip_file.stat().st_size / 1024:.1f} KB")

SUBMISSION READY
ZIP: ./submissions/tashkeel_700m_ft_submission_20260224_081824.zip
Size: 18.7 KB


In [18]:
# Sync to Google Drive
import subprocess
sync_script = PROJECT_DIR / 'sync_results.sh'
if sync_script.exists() and False:
    print("Syncing to Google Drive...")
    subprocess.run(['bash', str(sync_script)])

# Final cleanup - free GPU memory
print("\nCleaning up GPU memory...")
del model
del tokenizer
clear_gpu()
print("Done! GPU memory released.")

Syncing to Google Drive...
KSAA 2026 - Sync Results to Google Drive
  Project: .

[1/3] Syncing outputs...
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Elapsed time:         0.9sTransferred:   	  163.794 KiB / 163.794 KiB, 100%, 0 B/s, ETA -
Checks:                40 / 40, 100%
Transferred:            0 / 3, 0%
Elapsed time:         1.4s
Transferring:
 *         tashkeel_700m_ft_test_checkpoint.json:100% /83.004Ki, 0/s, -
 *        tashkeel_700m_ft_test_predictions.json:100% /79.628Ki, 0/s, -
 *           whisper_tashkeel_ft_checkpoint.json:100% /1.162Ki, 0/s, -Transferred:   	  163.794 KiB / 163.794 KiB, 100%, 0 B/s, ETA -
Checks:                40 / 40, 100%
Transferred:            0 / 3, 0%
Elapsed time:         1.9s
Transferring:
 *         tashkeel_700m_ft_test_checkpoint.json:100% /83.004Ki, 0/s, -
 *        tashkeel_700m_ft_test_predictions.json:100% /79.628Ki, 0/s, -
 *           whisper_tashkeel_ft_checkpoint.json:100% /1.162Ki, 0/s, -Transferred:   	  163.794 KiB / 16

2026/02/24 08:18:37 Failed to sync: corrupted on transfer: md5 hash differ "ebc892273f90a865485b35a8c7d7e8d9" vs "16439174ff6b9db24593cb1cc3e679ab"


GPU Memory: 2.10 GB allocated
Done! GPU memory released.
